# 🍌 Nano Banana Cloud — موتور تولید تصویر رایگان برای Minis

این نوت‌بوک یک سرور API پرسرعت و رایگان بر پایه **SDXL-Turbo / SDXL-Lightning** روی کارت گرافیک رایگان Google Colab (T4 GPU) اجرا می‌کند و از طریق تانل امن **Cloudflare** یک آدرس اینترنتی دائمی و بدون فیلتر در اختیارتان می‌گذارد.

### ✨ قابلیت‌ها:
- 🚀 **سازگار کامل با استاندارد OpenAI** ()
- ⚡ **سرعت فوق‌العاده**: تولید تصویر با کیفیت 1024x1024 در کمتر از ۲ ثانیه
- 🌐 **اتصال مستقیم به Minis**: کافیست آدرس تانل را در تنظیمات Minis یا اسکیل  قرار دهید
- 💰 **۱۰۰٪ رایگان و بدون محدودیت توکن**

---

In [ ]:
#@title 1. بررسی وضعیت کارت گرافیک (GPU Check)
!nvidia-smi


In [ ]:
# 2. نصب کتابخانه‌های مورد نیاز
!pip install -q fastapi uvicorn "diffusers>=0.28.0" transformers accelerate pycloudflared pillow requests
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ پیش‌نیازها نصب شدند!")


In [ ]:
# 3. ایجاد کدهای سرور (server.py)
code = '''
import os, time, io, base64, uuid, threading
from typing import Optional
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import HTMLResponse
from fastapi.staticfiles import StaticFiles
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import torch
from diffusers import AutoPipelineForText2Image

app = FastAPI(title="Nano Banana Colab API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
os.makedirs("outputs", exist_ok=True)
app.mount("/images", StaticFiles(directory="outputs"), name="images")

pipe = None
def load_engine():
    global pipe
    print("[*] Loading SDXL-Turbo on GPU...")
    pipe = AutoPipelineForText2Image.from_pretrained(
        "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16"
    ).to("cuda")
    print("[+] Ready to generate images!")

class ImageGenRequest(BaseModel):
    prompt: str
    model: Optional[str] = "nano-banana"
    size: Optional[str] = "1024x1024"
    response_format: Optional[str] = "b64_json"
    negative_prompt: Optional[str] = None
    seed: Optional[int] = None

@app.on_event("startup")
async def startup():
    threading.Thread(target=load_engine, daemon=True).start()

@app.get("/health")
def health():
    return {"status": "ready" if pipe is not None else "loading", "gpu": torch.cuda.get_device_name(0)}

@app.get("/v1/models")
def models():
    return {"object": "list", "data": [{"id": "nano-banana"}, {"id": "sdxl-turbo"}, {"id": "gemini-imagen"}]}

@app.post("/v1/images/generations")
async def gen(req: ImageGenRequest, request: Request):
    if pipe is None:
        raise HTTPException(503, "Engine still loading...")
    w, h = 1024, 1024
    if req.size and "x" in req.size:
        parts = req.size.split("x")
        w, h = min(int(parts[0]), 1024), min(int(parts[1]), 1024)
    generator = torch.Generator("cuda").manual_seed(req.seed) if req.seed else None
    with torch.inference_mode():
        img = pipe(prompt=req.prompt, num_inference_steps=2, guidance_scale=0.0, width=w, height=h, generator=generator).images[0]
    fname = f"{uuid.uuid4().hex[:12]}.png"
    img.save(os.path.join("outputs", fname))
    url = f"{str(request.base_url).rstrip('/')}/images/{fname}"
    item = {"url": url}
    if req.response_format != "url":
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        item["b64_json"] = base64.b64encode(buf.getvalue()).decode()
    return {"created": int(time.time()), "data": [item]}

@app.get("/", response_class=HTMLResponse)
def home():
    return "<h2>🍌 Nano Banana Engine Running!</h2><p>OpenAI compatible API at /v1/images/generations</p>"
'''
with open("server.py", "w") as f:
    f.write(code)
print("✅ فایل server.py ذخیره شد.")


In [ ]:
# 4. اجرای سرور و راه‌اندازی تانل کلودفلر (Cloudflare Tunnel)
import subprocess, time, re

# Kill any old instances
!pkill -f "uvicorn" || true
!pkill -f "cloudflared" || true

print("🚀 در حال استارت سرور uvicorn روی پورت 8000...")
server_proc = subprocess.Popen(["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

print("🌐 در حال ایجاد تانل امن کلودفلر...")
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stderr=subprocess.PIPE, text=True)

public_url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("
" + "="*60)
    print("🎉 تبریک! موتور Nano Banana با موفقیت روشن شد!")
    print(f"🔗 آدرس عمومی API: {public_url}/v1")
    print(f"🖼️ آدرس تست مستقیم در مرورگر: {public_url}")
    print("="*60)
    print("
📌 دستور تنظیم فوری در Minis:")
    print(f"minis-config set providers.65401b6b-7d1f-44ad-bbe4-00e1f4d654de.customBaseURL {public_url}/v1")
else:
    print("❌ در دریافت آدرس تانل تاخیر ایجاد شد. دوباره سلول را اجرا کنید.")


In [ ]:
# 5. نگه‌داشتن سرور (Keep-Alive)
import time
print("سشن فعال است و به درخواست‌های Minis پاسخ می‌دهد...")
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("متوقف شد.")
